# DDO-RM workshop notebook

One notebook, run **top to bottom**. Contains three diagnostics that share a
single model-loading step (so the GPU holds only one copy):

- **E1** — coverage → *realized* out-of-sample gain (the non-circular test).
- **E2** — full-on-support vs sampled-on-support (variance).
- **E3** — RM-argmax / Best-of-K ablation ("is DDO-RM just reranking?").

**How to use.** Set `MOCK_MODE` in the config cell: `True` runs everything on
CPU in seconds with synthetic stand-ins (positive controls that prove the
analysis works); `False` runs Pythia-410M + a real reward model on a GPU.

If you switch `MOCK_MODE` after already running, **restart the runtime** first
so stale globals / GPU memory are cleared, then run top to bottom.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Colab: uncomment once.
# !pip -q install transformers datasets accelerate peft sentencepiece scikit-learn matplotlib
# If you still hit "incompatible torchao" from peft, the backend cell auto-patches
# it; the clean alternative is:  !pip -q uninstall -y torchao   (then restart runtime).
print(">>> notebook build = ddorm-notebook-v3  (if you don't see this, wrong cell)")

## Config — set `MOCK_MODE` here

In [ ]:
# ===== global =====
MOCK_MODE   = True        # True = CPU synthetic. False = real Pythia-410m + RM (GPU).
MODEL_ID    = "EleutherAI/pythia-410m"
DATASET_ID  = "HuggingFaceH4/ultrafeedback_binarized"
REWARD_KIND = "rm"        # "rm" | "verifiable" | "logprob_CIRCULAR"
RM_ID       = "OpenAssistant/reward-model-deberta-v3-large-v2"
MAX_NEW_TOK = 96

# ===== E1 (coverage -> realized gain) =====
LAMBDA_GRID = [0.5, 1.0, 2.0]
M           = 16
K_VALUES    = [2, 4, 8]
RULES       = ["random", "coverage", "diverse"]
SEEDS       = [0, 1, 2]
N_TRAIN     = 48
N_HELDOUT   = 48
EVAL_SAMPLES= 4
FT_STEPS    = 40
FT_LR       = 1e-4        # LoRA likes a higher LR than full FT
FT_BATCH    = 4
LORA_R      = 8

# ===== E2 (full vs sampled) =====
E2_LAMBDA   = 1.0
E2_TAU      = 2.0         # softens the restricted policy p0 = softmax(scores/tau)
E2_K        = 8
E2_NPROMPTS = 64          # (mock uses synthetic supports; real generates this many)
E2_STEPS    = 60
E2_LR       = 0.3
E2_SEEDS    = [42, 13, 3407]

# ===== E3 (rerank ablation) =====
E3_LAMBDA   = 1.0
E3_K_EVAL   = [2, 4, 8]
E3_N_TRAIN  = 400         # mock; real reuses N_TRAIN / N_HELDOUT prompts
E3_N_EVAL   = 400
E3_D        = 16          # mock feature dim
E3_EPOCHS   = 200         # mock training epochs
E3_LR       = 0.2
E3_NOISE    = 0.3
E3_SEEDS    = [42, 13, 3407]

## Shared core (exact NumPy math, model-agnostic)

In [ ]:
import math, contextlib, numpy as np
from dataclasses import dataclass
from sklearn.feature_extraction.text import TfidfVectorizer

def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True); e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

# ---- tilt / coverage / soft value (E1) ----
def tilt_and_coverage(log_pi, reward, lam):
    log_w = reward / lam                      # 'is' mode (self-normalized IS)
    q = np.exp(log_w - log_w.max()); q /= q.sum()
    return q, log_w
def soft_value(log_w, idx, lam):
    lw = log_w[idx]; m = lw.max()
    return lam * (m + math.log(np.exp(lw - m).sum()))
def support_coverage(q, idx): return float(np.asarray(q)[idx].sum())
def support_gap(cov, lam=1.0): return lam * np.log(1.0 / max(cov, 1e-12))
def entropy(q):
    p = q[q > 0]; return float(-(p * np.log(p)).sum())

# ---- E2 objective + gradients (same objective, different estimator) ----
def restricted_policy(scores, tau=1.0): return softmax(scores / tau)
def grad_full(p, r, p0, lam=1.0):
    delta = np.log(np.clip(p,1e-12,None)) - np.log(np.clip(p0,1e-12,None))
    rbar = (p*r).sum(); klbar = (p*delta).sum()
    return p * ((r - rbar) - lam * (delta - klbar))
def grad_sampled(p, r, p0, lam, j):
    dj = np.log(np.clip(p[j],1e-12,None)) - np.log(np.clip(p0[j],1e-12,None))
    e = np.zeros_like(p); e[j] = 1.0
    return (r[j] - lam*dj) * (e - p)
def grad_sampled_variance(p, r, p0, lam):
    gf = grad_full(p, r, p0, lam); v = 0.0
    for j in range(len(p)):
        v += p[j] * float(((grad_sampled(p,r,p0,lam,j) - gf)**2).sum())
    return v
def kl_categorical(p, q):
    return float((p*(np.log(np.clip(p,1e-12,None))-np.log(np.clip(q,1e-12,None)))).sum())

# ---- baselines / metrics (E3) ----
def rm_argmax(r): return int(np.argmax(r))
def pair_accuracy(chosen, rejected):
    return float((np.asarray(chosen) > np.asarray(rejected)).mean())

# ---- selection + diversity (E1) ----
def embed(responses):
    try:
        X = TfidfVectorizer(min_df=1).fit_transform(responses).toarray().astype(float)
    except ValueError:
        X = np.eye(len(responses))
    n = np.linalg.norm(X,axis=1,keepdims=True); n[n==0]=1.0
    return X / n
def set_diversity(embeds, idx):
    if len(idx) < 2: return 0.0
    S = embeds[idx] @ embeds[idx].T; iu = np.triu_indices(len(idx),1)
    return float((1.0 - S[iu]).mean())
def select(rule, q, embeds, K, rng):
    n = len(q)
    if rule == "full" or K >= n: return np.arange(n)
    if rule == "random":   return rng.choice(n, size=K, replace=False)
    if rule == "coverage": return np.argsort(-q)[:K]
    if rule == "diverse":
        chosen=[int(np.argmax(q))]; S=embeds@embeds.T
        while len(chosen)<K:
            nn=S[:,chosen].max(1); nn[chosen]=np.inf; chosen.append(int(np.argmin(nn)))
        return np.array(chosen)
    raise ValueError(rule)
print("core loaded")

## Model backend (loaded once, shared by E1/E2/E3)

Real mode loads **one** frozen Pythia-410M with a single LoRA adapter (base =
adapter OFF, updated policy = adapter ON) plus the reward model. The updates
train the ~few-MB adapter, not a full copy, so it stays well under GPU memory.
Mock mode defines the same function names with synthetic stand-ins.

In [ ]:
if not MOCK_MODE:
    import torch, torch.nn.functional as F
    from transformers import (AutoTokenizer, AutoModelForCausalLM,
                              AutoModelForSequenceClassification)
    from peft import LoraConfig, get_peft_model
    import sys as _sys                        # workaround: on Colab peft's torchao
    for _n,_m in list(_sys.modules.items()):  # version check raises on old torchao.
        if _n.startswith("peft") and hasattr(_m, "is_torchao_available"):
            setattr(_m, "is_torchao_available", lambda: False)   # disable that check
    from datasets import load_dataset

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    _base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).to(DEVICE)
    for p in _base.parameters(): p.requires_grad_(False)
    try:
        _cfg = LoraConfig(r=LORA_R, lora_alpha=2*LORA_R, lora_dropout=0.0,
                          target_modules="all-linear", task_type="CAUSAL_LM")
        _model = get_peft_model(_base, _cfg)
    except Exception:
        _cfg = LoraConfig(r=LORA_R, lora_alpha=2*LORA_R, lora_dropout=0.0,
                          target_modules=["query_key_value","dense"], task_type="CAUSAL_LM")
        _model = get_peft_model(_base, _cfg)
    _model.eval()

    def reset_lora():
        for m in _model.modules():
            if hasattr(m, "lora_A"):
                for k in m.lora_A: torch.nn.init.kaiming_uniform_(m.lora_A[k].weight, a=math.sqrt(5))
                for k in m.lora_B: torch.nn.init.zeros_(m.lora_B[k].weight)
    def _ctx(adapter): return contextlib.nullcontext() if adapter else _model.disable_adapter()

    if REWARD_KIND == "rm":
        _rmtok = AutoTokenizer.from_pretrained(RM_ID)
        _rm = AutoModelForSequenceClassification.from_pretrained(RM_ID, dtype=torch.float16).to(DEVICE).eval()
        @torch.no_grad()
        def reward_fn(prompt, responses, gold=None):
            out=[]
            for r in responses:
                enc=_rmtok(prompt, r, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
                out.append(float(_rm(**enc).logits[0,0].item()))
            return np.array(out, dtype=np.float64)
    elif REWARD_KIND == "verifiable":
        import re
        def _ext(t):
            m=re.findall(r"-?\d+\.?\d*", t.replace(",","")); return m[-1] if m else None
        def reward_fn(prompt, responses, gold=None):
            return np.array([1.0 if _ext(r)==gold else 0.0 for r in responses])
    else:  # logprob_CIRCULAR (smoke test only)
        def reward_fn(prompt, responses, gold=None):
            return seq_logprob(prompt, responses, adapter=False, length_normalize=True)

    @torch.no_grad()
    def gen_pool(prompt, n, adapter=False, max_new_tokens=MAX_NEW_TOK):
        inp=tokenizer(prompt, return_tensors="pt").to(DEVICE); plen=inp["input_ids"].shape[1]
        with _ctx(adapter):
            out=_model.generate(**inp, do_sample=True, temperature=0.9, top_p=0.95,
                                num_return_sequences=n, max_new_tokens=max_new_tokens,
                                pad_token_id=tokenizer.eos_token_id)
        return tokenizer.batch_decode(out[:, plen:], skip_special_tokens=True)

    @torch.no_grad()
    def seq_logprob(prompt, responses, adapter=False, length_normalize=False):
        pid=tokenizer(prompt, return_tensors="pt")["input_ids"][0]; out=[]
        with _ctx(adapter):
            for resp in responses:
                rid=tokenizer(resp, return_tensors="pt")["input_ids"][0]
                if rid.numel()==0: out.append(-1e4); continue
                ids=torch.cat([pid,rid]).unsqueeze(0).to(DEVICE)
                lg=F.log_softmax(_model(ids).logits[0,:-1].float(), -1); tgt=ids[0,1:]
                lp=lg[torch.arange(tgt.numel()), tgt][-rid.numel():].sum().item()
                out.append(lp/rid.numel() if length_normalize else lp)
        return np.array(out, dtype=np.float64)

    def mirror_update(items, steps=FT_STEPS, lr=FT_LR):
        reset_lora(); _model.train()
        params=[p for p in _model.parameters() if p.requires_grad]
        opt=torch.optim.AdamW(params, lr=lr)
        flat=[(it["prompt"], y, float(w)) for it in items for y,w in zip(it["responses"], it["weights"])]
        rng=np.random.default_rng(0)
        for _ in range(steps):
            idx=rng.integers(0, len(flat), size=min(FT_BATCH, len(flat))); opt.zero_grad()
            for j in idx:
                prompt,y,w=flat[j]
                pid=tokenizer(prompt, return_tensors="pt")["input_ids"][0]
                rid=tokenizer(y, return_tensors="pt")["input_ids"][0]
                if rid.numel()==0: continue
                ids=torch.cat([pid,rid]).unsqueeze(0).to(DEVICE)
                lg=F.log_softmax(_model(ids).logits[0,:-1].float(), -1); tgt=ids[0,1:]
                nll=-lg[torch.arange(tgt.numel()), tgt][-rid.numel():].mean()
                (w*nll/len(idx)).backward()
            opt.step()
        _model.eval(); return _model

    def eval_value(heldout, adapter, n_samples=EVAL_SAMPLES):
        vals=[]
        for ex in heldout:
            samp=gen_pool(ex["prompt"], n_samples, adapter=adapter)
            r=reward_fn(ex["prompt"], samp, ex.get("answer")) if REWARD_KIND=="verifiable" else reward_fn(ex["prompt"], samp)
            vals.append(float(np.mean(r)))
        torch.cuda.empty_cache(); return float(np.mean(vals))

    def load_prompts(n_train, n_held):
        ds=load_dataset(DATASET_ID, split="test_prefs")
        pr=[(p if isinstance(p,str) else p[0]) for p in ds["prompt"]]
        return ([{"prompt":p} for p in pr[:n_train]],
                [{"prompt":p} for p in pr[n_train:n_train+n_held]])
    print("real backend ready on", DEVICE)
else:
    # ---------------- MOCK backend (CPU) ----------------
    _MOCK={"quality":0.0}; _MRNG=np.random.default_rng(123)
    def _latent(prompt, n):
        h=abs(hash(prompt))%(2**31); return np.random.default_rng(h).normal(0,1,n)
    def load_prompts(n_train, n_held):
        return ([{"prompt":f"train_{i}"} for i in range(n_train)],
                [{"prompt":f"held_{i}"} for i in range(n_held)])
    def gen_pool(prompt, n, adapter=False, max_new_tokens=MAX_NEW_TOK):
        return [f"{prompt}::c{i}::{'x '*((i*7)%9)}" for i in range(n)]
    def reward_fn(prompt, responses, gold=None):
        return _latent(prompt, len(responses))
    def seq_logprob(prompt, responses, adapter=False, length_normalize=False):
        return _latent(prompt, len(responses))*0.3
    def mirror_update(items, steps=FT_STEPS, lr=FT_LR):
        caps=[]
        for it in items:
            w=np.asarray(it["weights"]); w=w/w.sum()
            caps.append(float(it.get("rho",0.0)) + 0.05*float((w*np.asarray(it["rewards"])).sum()))
        _MOCK["quality"]=float(np.mean(caps)); return _MOCK
    def eval_value(heldout, adapter, n_samples=EVAL_SAMPLES):
        return (_MOCK["quality"]+float(_MRNG.normal(0,0.02))) if adapter else 0.0
    print("mock backend ready (CPU)")

## E1 — coverage → realized out-of-sample gain

Non-circular test. For each selected support C: build the reward-tilt, do a real
mirror update (reward-weighted LoRA fine-tune restricted to C), then measure the
updated policy's value on **disjoint** held-out prompts. Horse race: does ρ
predict realized gain better than K / entropy / reward-var / max-reward /
diversity? Plus: does in-sample `λ log(1/ρ)` track the *measured* restriction
cost `J_full − J_C`?

In [ ]:
@dataclass
class PromptPack:
    prompt: str; responses: list; reward: np.ndarray; log_pi: np.ndarray; embeds: np.ndarray

def e1_build_packs(train):
    packs=[]
    for ex in train:
        resp=gen_pool(ex["prompt"], M, adapter=False)
        rwd=reward_fn(ex["prompt"], resp, ex.get("answer")) if REWARD_KIND=="verifiable" else reward_fn(ex["prompt"], resp)
        lp=seq_logprob(ex["prompt"], resp, adapter=False)
        packs.append(PromptPack(ex["prompt"], resp, rwd, lp, embed(resp)))
    return packs

def e1_run():
    train, held = load_prompts(N_TRAIN, N_HELDOUT)
    J0 = eval_value(held, adapter=False)
    rows=[]
    for seed in SEEDS:
        srng=np.random.default_rng(seed); packs=e1_build_packs(train)
        for lam in LAMBDA_GRID:
            tilts=[tilt_and_coverage(p.log_pi, p.reward, lam) for p in packs]; J_full=None
            for rule in RULES+["full"]:
                for K in ([M] if rule=="full" else K_VALUES):
                    if K>M: continue
                    items,rhos,gaps=[],[],[]; sm,sx,sv,sd=[],[],[],[]
                    for p,(q,log_w) in zip(packs, tilts):
                        C=select(rule,q,p.embeds,K,srng); rho=float(q[C].sum()); w=q[C]/q[C].sum()
                        items.append({"prompt":p.prompt,"responses":[p.responses[i] for i in C],
                                      "weights":w,"rho":rho,"rewards":p.reward[C]})
                        rhos.append(rho)
                        gaps.append(soft_value(log_w,np.arange(M),lam)-soft_value(log_w,C,lam))
                        sm.append(float(p.reward[C].mean())); sx.append(float(p.reward[C].max()))
                        sv.append(float(p.reward[C].var()));  sd.append(set_diversity(p.embeds,C))
                    mirror_update(items); J=eval_value(held, adapter=True)
                    if rule=="full": J_full=J
                    rows.append(dict(seed=seed,lam=lam,rule=rule,K=K,rho=float(np.mean(rhos)),
                        neg_log_rho=float(np.mean([math.log(1/max(r,1e-12)) for r in rhos])),
                        gap_insample=float(np.mean(gaps)),set_mean_reward=float(np.mean(sm)),
                        set_max_reward=float(np.mean(sx)),set_reward_var=float(np.mean(sv)),
                        set_diversity=float(np.mean(sd)),
                        pool_entropy=float(np.mean([entropy(q) for q,_ in tilts])),
                        K_pred=K,J=J,J0=J0,realized_gain=J-J0))
            for r in rows:
                if r["seed"]==seed and r["lam"]==lam and r["rule"]!="full":
                    r["restriction_cost_measured"]=(J_full-r["J"]) if J_full is not None else np.nan
    return rows

def e1_analyze(rows):
    import pandas as pd
    df=pd.DataFrame(rows)
    print(f"MOCK_MODE={MOCK_MODE} reward={REWARD_KIND} configs={len(df)} J0={df.J0.iloc[0]:.3f}\n")
    sub=df[df.rule!="full"].copy()
    preds=["rho","neg_log_rho","K_pred","set_mean_reward","set_max_reward","set_reward_var","set_diversity","pool_entropy"]
    print("=== HORSE RACE: predictors of realized gain (Pearson r) ===")
    corr={p:(float("nan") if (sub[p].nunique()<2 or sub[p].std()==0) else sub[p].corr(sub["realized_gain"])) for p in preds}
    for k,v in sorted(corr.items(), key=lambda kv:(-abs(kv[1]) if kv[1]==kv[1] else 1)):
        print(f"  {k:16s}: r = {v:+.3f}" if v==v else f"  {k:16s}: r =   n/a")
    if sub["restriction_cost_measured"].notna().any():
        s=sub.dropna(subset=["restriction_cost_measured"])
        r=s["gap_insample"].corr(s["restriction_cost_measured"]); b=np.polyfit(s["gap_insample"],s["restriction_cost_measured"],1)
        print(f"\n=== IDENTITY AS PREDICTION: lambda*log(1/rho) vs measured (J_full-J_C) ===")
        print(f"  Pearson r={r:+.3f}  slope={b[0]:+.3f}  (1.0=holds out-of-sample; <1=attenuated)")
    print("\n=== realized gain by rule x K ===")
    print(sub.groupby(["rule","K"])[["rho","realized_gain"]].mean().round(3))
    return df

e1_df = e1_analyze(e1_run())

## E2 — full-on-support vs sampled-on-support

Both rules ascend the *same* objective `J(p)=E_p[r] − λ·KL(p‖p0)`; only the
gradient estimator differs (`full` uses all K rewards → deterministic;
`sampled` uses one draw → unbiased but high-variance). Fair comparison =
reward at **matched KL**. No LM fine-tuning — the K-way categorical is trained
in NumPy — so this cell is light on memory even in real mode.

In [ ]:
_E2_CACHE={"sup":None}
def e2_supports():
    if MOCK_MODE:
        rng=np.random.default_rng(0)
        return [(rng.normal(0,1.0,E2_K), rng.normal(0,1.5,E2_K)) for _ in range(E2_NPROMPTS)]
    if _E2_CACHE["sup"] is not None: return _E2_CACHE["sup"]
    train,_=load_prompts(E2_NPROMPTS, 0); sup=[]
    for ex in train:
        resp=gen_pool(ex["prompt"], E2_K, adapter=False)
        s=seq_logprob(ex["prompt"], resp, adapter=False)            # raw seq log-probs
        s=(s-s.mean())/(s.std()+1e-6)                               # standardize -> non-degenerate p0
        r=reward_fn(ex["prompt"], resp, ex.get("answer")) if REWARD_KIND=="verifiable" else reward_fn(ex["prompt"], resp)
        sup.append((s, r))
    _E2_CACHE["sup"]=sup; return sup

def e2_train(s, r, rule, rng):
    z=s.copy()/E2_TAU; p0=softmax(z); traj=[]
    for _ in range(E2_STEPS):
        p=softmax(z); traj.append((float((p*r).sum()), kl_categorical(p,p0),
                                   grad_sampled_variance(p,r,p0,E2_LAMBDA)))
        if rule=="full": g=grad_full(p,r,p0,E2_LAMBDA)
        else:            g=grad_sampled(p,r,p0,E2_LAMBDA, int(rng.choice(len(p), p=p)))
        z=z+E2_LR*g
    return np.array(traj)

def e2_run():
    sup=e2_supports(); rows,curves=[],[]
    for seed in E2_SEEDS:
        for rule in ["full","sampled"]:
            rng=np.random.default_rng(seed+7)
            trajs=np.stack([e2_train(s,r,rule,rng) for s,r in sup]); mt=trajs.mean(0)
            for t in range(E2_STEPS):
                curves.append(dict(seed=seed,rule=rule,step=t,reward=mt[t,0],kl=mt[t,1],grad_var=mt[t,2]))
            f=trajs[:,-1,:]
            rows.append(dict(seed=seed,rule=rule,final_reward=float(f[:,0].mean()),
                final_kl=float(f[:,1].mean()),
                mean_grad_var=float(trajs[:,:,2].mean() if rule=="sampled" else 0.0)))
    import pandas as pd
    return pd.DataFrame(rows), pd.DataFrame(curves)

def e2_analyze(summary, curves):
    import pandas as pd
    print(f"MOCK_MODE={MOCK_MODE}  K={E2_K}  prompts={E2_NPROMPTS}  tau={E2_TAU}\n")
    print("=== end-of-run (mean over seeds) ===")
    print(summary.groupby("rule")[["final_reward","final_kl","mean_grad_var"]].mean().round(4),"\n")
    kmax=min(curves[curves.rule==r]["kl"].max() for r in ["full","sampled"])  # overlap only
    kl_grid=np.linspace(0.02, max(0.05, 0.9*kmax), 10); out=[]
    for rule in ["full","sampled"]:
        sub=curves[curves.rule==rule].groupby("step")[["reward","kl"]].mean().sort_values("kl")
        for kl,rw in zip(kl_grid, np.interp(kl_grid, sub["kl"], sub["reward"])):
            out.append(dict(rule=rule,kl=round(float(kl),3),reward=float(rw)))
    piv=pd.DataFrame(out).pivot(index="kl",columns="rule",values="reward")
    piv["full_minus_sampled"]=(piv["full"]-piv["sampled"]).round(4)
    print("=== reward at MATCHED KL (full should be >= sampled) ==="); print(piv.round(4))
    return piv

e2_summary, e2_curves = e2_run()
e2_piv = e2_analyze(e2_summary, e2_curves)

## E3 — RM-argmax / Best-of-K ablation

"Is DDO-RM just reranking?" Train the policy, then evaluate the **trained policy
on NEW prompts / NEW candidates WITHOUT the RM at inference**. If DDO-RM ≫ base
on unseen prompts (approaching RM-argmax, which *does* use the RM at test), it
internalized the reward. Mock = shared linear model (validates generalization on
CPU). Real = DDO-RM LoRA distillation + held-out pick eval with the real RM.

Honest expectation on the real 410M run: DDO-RM should land *between* base and
RM-argmax, not at the oracle. That is still a valid, publishable result.

In [ ]:
def e3_run():
    rows=[]
    if MOCK_MODE:
        for seed in E3_SEEDS:
            rng=np.random.default_rng(seed); w_true=rng.normal(0,1,E3_D)
            def mk(n,k):
                out=[]
                for _ in range(n):
                    Phi=rng.normal(0,1,(k,E3_D)); r=Phi@w_true+rng.normal(0,E3_NOISE,k); out.append((Phi,r))
                return out
            train=mk(E3_N_TRAIN, 8); theta=np.zeros(E3_D)
            for _ in range(E3_EPOCHS):
                g=np.zeros(E3_D)
                for Phi,r in train:
                    p=softmax(Phi@theta); q=softmax(r/E3_LAMBDA); g+=Phi.T@(p-q)
                theta-=E3_LR*g/len(train)
            theta0=rng.normal(0,1,E3_D)
            for k in E3_K_EVAL:
                ev=mk(E3_N_EVAL,k); acc={m:[] for m in ["base","ddorm","rm","rand"]}; pa_b,pa_d=[],[]
                rr=np.random.default_rng(0)
                for Phi,r in ev:
                    acc["base"].append(r[int(np.argmax(Phi@theta0))]); acc["ddorm"].append(r[int(np.argmax(Phi@theta))])
                    acc["rm"].append(r[rm_argmax(r)]); acc["rand"].append(r[rr.integers(len(r))])
                    hi,lo=int(np.argmax(r)),int(np.argmin(r))
                    pa_b.append((Phi[hi]@theta0)>(Phi[lo]@theta0)); pa_d.append((Phi[hi]@theta)>(Phi[lo]@theta))
                lo_,hi_=np.mean(acc["rand"]),np.mean(acc["rm"])
                rows.append(dict(seed=seed,K_eval=k,reward_base=np.mean(acc["base"]),reward_ddorm=np.mean(acc["ddorm"]),
                    reward_rm=hi_,reward_random=lo_,
                    ddorm_frac=(np.mean(acc["ddorm"])-lo_)/(hi_-lo_+1e-9),
                    base_frac=(np.mean(acc["base"])-lo_)/(hi_-lo_+1e-9),
                    pair_acc_base=float(np.mean(pa_b)),pair_acc_ddorm=float(np.mean(pa_d))))
    else:
        train, held = load_prompts(N_TRAIN, N_HELDOUT)
        # distill q* (softmax(r/lambda)) over base-policy candidate pools into the LoRA policy
        items=[]
        for ex in train:
            resp=gen_pool(ex["prompt"], M, adapter=False)
            r=reward_fn(ex["prompt"], resp, ex.get("answer")) if REWARD_KIND=="verifiable" else reward_fn(ex["prompt"], resp)
            items.append({"prompt":ex["prompt"],"responses":resp,"weights":softmax(r/E3_LAMBDA),"rewards":r})
        mirror_update(items)
        rr=np.random.default_rng(0)
        for k in E3_K_EVAL:
            base_r,ddo_r,rm_r,rand_r=[],[],[],[]
            for ex in held:
                resp=gen_pool(ex["prompt"], k, adapter=False)     # fresh candidates from base
                r=reward_fn(ex["prompt"], resp, ex.get("answer")) if REWARD_KIND=="verifiable" else reward_fn(ex["prompt"], resp)
                lp_b=seq_logprob(ex["prompt"], resp, adapter=False)
                lp_d=seq_logprob(ex["prompt"], resp, adapter=True)
                base_r.append(r[int(np.argmax(lp_b))]); ddo_r.append(r[int(np.argmax(lp_d))])
                rm_r.append(r[rm_argmax(r)]); rand_r.append(r[rr.integers(len(r))])
            lo_,hi_=float(np.mean(rand_r)),float(np.mean(rm_r))
            rows.append(dict(seed=0,K_eval=k,reward_base=float(np.mean(base_r)),reward_ddorm=float(np.mean(ddo_r)),
                reward_rm=hi_,reward_random=lo_,
                ddorm_frac=(np.mean(ddo_r)-lo_)/(hi_-lo_+1e-9),base_frac=(np.mean(base_r)-lo_)/(hi_-lo_+1e-9),
                pair_acc_base=float("nan"),pair_acc_ddorm=float("nan")))
    return rows

def e3_analyze(rows):
    import pandas as pd
    df=pd.DataFrame(rows)
    print(f"MOCK_MODE={MOCK_MODE}  K_eval={E3_K_EVAL}\n")
    g=df.groupby("K_eval").mean(numeric_only=True).round(3)
    print("=== held-out selection quality on NEW prompts/candidates ===")
    print(g[["reward_random","reward_base","reward_ddorm","reward_rm"]],"\n")
    print("=== fraction of oracle captured [0=random,1=RM-argmax] ===")
    print(g[["base_frac","ddorm_frac"]],"\n")
    print("READ: ddorm_frac >> base_frac on NEW prompts => generalizes, not mere reranking.")
    print("RM-argmax/Best-of-K use the RM at test; DDO-RM does not.")
    return df

e3_df = e3_analyze(e3_run())

## Notes

- Flip `MOCK_MODE=False` (restart runtime first) for real results. Start small
  (`N_TRAIN=32`, `EVAL_SAMPLES=2`) to check wall-clock, then scale.
- The mocks are **positive controls**, not evidence — they only confirm each
  analysis can detect the effect when it's present.
- Reward must not be the policy's own log-prob for the real claims; use `"rm"`
  or `"verifiable"`. `"logprob_CIRCULAR"` reproduces the tautology on purpose.